# Quantum-circuit repair: open LLMs on a free Colab GPU

Runs the **LLM-only arm** of `physics-constrained-quantum-repair` for several open models served by Ollama.
The protocol is identical for every model: the model is the only repair engine (no oracle, no search, no fallback),
three attempts, validator feedback by stage name, temperature 0, fixed seed, same acceptance predicate.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. Run the cells top to bottom.
Everything is resumable: if Colab disconnects, run cells 1-3 again and then the run cells; finished circuits are kept
(the partial files live in `experiments/llm_runs/`, mirrored to Google Drive after every model).

Two kinds of run:
* **Comparison run (60 circuits)**: the same stratified 60 circuits used for every other LLM result, so models can be compared pairwise.
* **Large run (1000 circuits)**: one fast model on the whole benchmark, for a tight confidence interval.

No API key is needed here, and nothing secret is stored in this notebook.

## 1. Setup (about 3 minutes)

In [ ]:
import os, subprocess, textwrap
def sh(cmd, check=True):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr)[-3000:])
    if check and r.returncode: raise RuntimeError('command failed')

if not os.path.exists('/content/physics-constrained-quantum-repair'):
    sh('git clone --depth 1 https://github.com/alok-108/physics-constrained-quantum-repair.git /content/physics-constrained-quantum-repair')
os.chdir('/content/physics-constrained-quantum-repair')
sh('git log --oneline | head -1')
sh('pip install -q -r requirements.txt langchain-core')
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')

In [ ]:
# Ollama (the zstd package is needed by the installer)
sh('apt-get install -y -qq zstd >/dev/null')
if subprocess.run('which ollama', shell=True).returncode:
    sh('curl -fsSL https://ollama.com/install.sh | sh')
sh('ollama --version')

## 2. Configuration

In [ ]:
# Models for the 60-circuit comparison run. The first run used qwen2.5-coder:1.5b locally; it is not repeated here.
MODELS_60 = [
    'qwen2.5-coder:3b',
    'qwen2.5-coder:14b',
    'deepseek-coder-v2:16b',
    'llama3.1:8b',
    'gemma3:12b',
    'phi4:14b',
    'mistral:7b',
]
# One fast model for the full 1000-circuit run (it is also scored on the shared 60 by the analysis script).
MODEL_1000 = 'qwen2.5-coder:7b'
WORKERS = 4            # parallel circuits; the server is told to serve this many requests at once
DRIVE_DIR = '/content/drive/MyDrive/qrepair_llm_runs'   # results are copied here after every model

env = dict(os.environ, OLLAMA_NUM_PARALLEL=str(WORKERS), OLLAMA_CONTEXT_LENGTH='4096',
           OLLAMA_KEEP_ALIVE='30m', PYTHONUNBUFFERED='1')

In [ ]:
# Optional but recommended: keep the results on your Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
except Exception as e:
    print('Drive not mounted:', e); DRIVE_DIR = None

In [ ]:
import time, urllib.request
def start_server():
    try:
        urllib.request.urlopen('http://localhost:11434', timeout=2); return
    except Exception:
        pass
    subprocess.Popen('ollama serve > /content/ollama.log 2>&1', shell=True, env=env)
    for _ in range(60):
        time.sleep(1)
        try:
            urllib.request.urlopen('http://localhost:11434', timeout=2); print('ollama is up'); return
        except Exception:
            pass
    raise RuntimeError('ollama did not start; see /content/ollama.log')
start_server()

## 3. Run one model
`run_model(tag, limit)` pulls the model, runs the LLM-only arm on the stratified circuits, saves the CSV to
`experiments/llm_runs/` (and Drive), then removes the model to free disk space.

In [ ]:
import glob, shutil
def run_model(model, limit):
    tag = model.replace(':', '_').replace('/', '_') + ('' if limit == 60 else f'_{limit}')
    out = f'experiments/llm_runs/{tag}.csv'
    os.makedirs('experiments/llm_runs', exist_ok=True)
    if os.path.exists(out):
        print('already done:', out); return out
    r = subprocess.run(f'ollama pull {model}', shell=True, text=True, capture_output=True)
    if r.returncode:
        print('SKIPPED (pull failed):', model, r.stderr[-300:]); return None
    e = dict(env, QREPAIR_OLLAMA_MODEL=model)
    cmd = (f'python src/run_experiments.py --metadata data/metadata.csv --out {out} --limit {limit} '
           f'--systems llm_ollama --workers {WORKERS} --resume')
    print('$', cmd)
    t = time.time()
    p = subprocess.Popen(cmd, shell=True, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'circuits (' in line or 'Wrote' in line or 'STOPPED' in line or 'Error' in line: print(line.rstrip())
    p.wait()
    print(f'{model}: finished in {(time.time() - t) / 60:.1f} min')
    if DRIVE_DIR:
        for f in glob.glob('experiments/llm_runs/*'): shutil.copy(f, DRIVE_DIR)
    subprocess.run(f'ollama rm {model}', shell=True)
    return out

## 4. Comparison runs: 60 circuits per model

In [ ]:
for m in MODELS_60:
    run_model(m, 60)

## 5. Large run: 1000 circuits, one model
On a T4 with 4 parallel requests a 7B model should take a few hours. If the session is cut off, run cells 1-3 and this cell again;
`--resume` continues from the saved circuits. For a quicker run set `MODEL_1000 = 'qwen2.5-coder:3b'` in the configuration cell.

In [ ]:
run_model(MODEL_1000, 1000)

## 6. Download the results
Send me the zip (or tell me the Drive folder). I merge the CSVs into the repository and regenerate every table, the paper and the website.
Put them in `experiments/llm_runs/` and run `python src/analyze_llm.py`.

In [ ]:
sh('cd experiments && zip -qr /content/llm_runs.zip llm_runs -x "*.partial.jsonl"')
try:
    from google.colab import files
    files.download('/content/llm_runs.zip')
except Exception as e:
    print('zip is at /content/llm_runs.zip', e)